# 03b — E6 boundary-aware ConvNeXt-Tiny (Study 03 follow-up)

**Status: IMPLEMENTED, NOT RUN.** This is an exploratory follow-up to E5, not a fourth primary study. No E6 cache, A100 benchmark, training, validation result or final test is claimed here. Every execution control below defaults to `False`. Read the [E6 protocol](../novelty_integrated/e6/README.md) before using an authorized HPC session.

## 1. Background, question and evidence status

The [CAD-ViT paper](https://doi.org/10.1109/ACCESS.2025.3572455) studies multi-scale retinal feature fusion and attention; it does **not** define this E6 objective. The E5 ConvNeXt study uses a categorical classifier plus ordinal auxiliary head and delayed inverse-square-root weighting. The operator reported DDR E5 validation Grade 1 correct counts of 5/63, 4/63, 4/63 and 0/63 at epochs 6, 8, 9 and 10, with predictions shifting between Grade 0 and Grade 2. Those interim numbers are operator-reported, not independently verified in this checkout and not E6 results. They motivate a **hypothesis** that explicitly training adjacent categorical boundaries could help Grade 1. They cannot establish whether image resolution, annotation ambiguity, representation quality or class imbalance caused the errors.

## 2. Literature and mechanism

[Cao et al.](https://proceedings.nips.cc/paper_files/paper/2019/hash/621461af90cadfdaf0e8d4cc25129f91-Abstract.html) introduce LDAM and deferred reweighting; [Cui et al.](https://openaccess.thecvf.com/content_CVPR_2019/html/Cui_Class-Balanced_Loss_Based_on_Effective_Number_of_Samples_CVPR_2019_paper.html) define effective-number weighting. [Stelter et al.](https://proceedings.mlr.press/v307/stelter26a.html) examine a different distribution-based ordinal loss for DR; E6 does not reproduce it. E6 uses the existing E5 monotonic threshold head and adds a small pairwise softplus term for the 0–1 and 1–2 categorical boundaries. LDAM and the pairwise term both influence the same logits, so their incremental value is **unconfirmed**. A later controlled ablation and targeted prior-art search are needed before a methodological novelty claim.

## 3. Safe environment and provenance

The source is `novelty_integrated.e6`; the official ConvNeXt-Tiny snapshot is locked by [`config.py`](../novelty_integrated/e6/config.py). The frozen seed-42 manifest and JSON sidecar live outside Git on authorized HPC storage. This notebook never rewrites them. DDR official source directory names do not override frozen `split` assignments. All `split=test` rows are skipped by the imported loader before paths/labels are opened. APTOS and DDR are independent runs.

In [ ]:
from pathlib import Path
import os, json, hashlib, importlib.util
import sys
_candidate = Path(os.environ.get("DR_RESEARCH_ROOT", ".")).expanduser().resolve()
REPO = next((p for p in (_candidate, _candidate.parent) if (p / "novelty_integrated" / "e6" / "runner.py").is_file()), _candidate)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ARTIFACT = Path(os.environ.get("DR_ARTIFACT_ROOT", "")).expanduser() if os.environ.get("DR_ARTIFACT_ROOT") else None
OUTPUT = Path(os.environ.get("DR_E6_OUTPUT_ROOT", "")).expanduser() if os.environ.get("DR_E6_OUTPUT_ROOT") else None
DATASET = os.environ.get("DR_E6_DATASET", "DDR")
RUN_PREFLIGHT = False
RUN_CACHE_BUILD = False
RUN_SMOKE = False
RUN_BENCHMARK = False
RUN_TRAINING = False
print({"repository": str(REPO), "dataset": DATASET, "artifact_root_set": ARTIFACT is not None,
       "output_root_set": OUTPUT is not None, "torch_available": importlib.util.find_spec("torch") is not None,
       "all_execution_controls_off": not any((RUN_PREFLIGHT,RUN_CACHE_BUILD,RUN_SMOKE,RUN_BENCHMARK,RUN_TRAINING))})

### Frozen split and source hash gate

The next cell reads only local manifest CSV/sidecar bytes if present. It does not open images or access the frozen test split. The runner later verifies **every** train/validation source image SHA before cache admission or model use.

In [ ]:
if ARTIFACT is None or DATASET not in ("APTOS2019", "DDR"):
    print("SKIPPED_MISSING_DATA: set DR_ARTIFACT_ROOT and DR_E6_DATASET")
else:
    manifest = ARTIFACT / "cad_vit_runs" / "manifests" / f"{DATASET}_seed42.csv"
    sidecar = manifest.with_suffix(".json")
    if not manifest.is_file() or not sidecar.is_file():
        print("SKIPPED_MISSING_DATA: original frozen manifest/sidecar unavailable")
    else:
        expected = json.loads(sidecar.read_text(encoding="utf-8"))
        digest = hashlib.sha256(manifest.read_bytes()).hexdigest()
        print({"dataset": DATASET, "manifest_sha256": digest,
               "matches_sidecar": digest == expected.get("manifest_file_sha256"),
               "seed": expected.get("seed"), "source_root_recorded": expected.get("root")})
        assert digest == expected["manifest_file_sha256"] and expected["seed"] == 42

## 4. Input protocol and optional train image view

Each train image is SHA-checked, EXIF-transposed, converted to RGB and stored as exact `uint8` pixels in the E6 external cache. On every epoch the existing B1 horizontal flip, conservative rotation, brightness and contrast transformations are drawn anew with the same image-keyed seed rule; the official processor then makes 224×224 model tensors. Validation uses the deterministic official processor, optionally via a **read-only** E5 cache after full compatibility checks. No augmentation output is stored. Raw fundus images are not embedded in this Git notebook; authorized users may enable the next local-only view for a verified training image.

In [ ]:
SHOW_VERIFIED_TRAIN_SAMPLE = False
if not SHOW_VERIFIED_TRAIN_SAMPLE:
    print("SKIPPED_BY_DEFAULT: no fundus image is loaded or redistributed")
else:
    from novelty_integrated.e5 import data
    from PIL import Image, ImageOps
    from IPython.display import display
    assert ARTIFACT is not None and DATASET in ("APTOS2019", "DDR")
    rows, info = data.load_train_val(ARTIFACT, DATASET,
        manifest_source_root=os.environ.get("DR_MANIFEST_SOURCE_ROOT"), verify_images=True)
    row = rows["train"][0]
    assert row["split"] == "train"
    with Image.open(row["path"]) as image:
        display(ImageOps.exif_transpose(image).convert("RGB").copy())

## 5. Architecture and mathematical objective

Official `facebook/convnext-tiny-224` → pooled 768-D feature → five-class categorical head and E5 monotonic four-threshold ordinal auxiliary head. The categorical head alone produces grades and evaluation probabilities. From train-only counts `n_c`, `m_c=0.5 n_c^{-1/4}/max_j n_j^{-1/4}` and `L_LDAM=CE(30(z-m_y e_y),y)`. Deferred effective-number weights at epoch 11 use `E(n)=(1-0.9999^n)/(1-0.9999)` and normalize mean train-image weight to one. Ordinal targets are `1[y>k]`. For `N_0={1},N_1={0,2},N_2={1}`, the adjacent term is the image-mean of `mean_{j∈N_y} softplus(30(z_j-z_y))/30`; grades 3/4 contribute zero. **Total:** `L_LDAM-DRW + 0.25 L_ordinal + 0.1 L_adjacent`. The last coefficient is a prespecified pilot assumption, not a literature optimum. The complete implementation is imported below; model definitions are not copied into this notebook.

In [ ]:
if importlib.util.find_spec("torch") is None:
    print("SKIPPED_MISSING_DATA: PyTorch is required for E6 implementation checks")
else:
    from novelty_integrated.e6 import config
    from novelty_integrated.e6.objective import loss_components
    from novelty_integrated.e5.model import E5Heads
    print({"schema": config.SCHEMA, "official_snapshot": config.OFFICIAL_SNAPSHOT["hub_id"],
           "revision": config.OFFICIAL_SNAPSHOT["revision"],
           "batch": config.EFFECTIVE_BATCH, "epochs": config.PILOT_EPOCHS,
           "boundary_coefficient": config.ADJACENT_COEFFICIENT})

## 6. Cache capacity, resource isolation and executable preflight

The commands are assembled from `DR_*` environment variables and are **printed first**. `RUN_PREFLIGHT` must be enabled deliberately in an authorized HPC notebook. CPU header sizing is separate from cache population. Inspect cgroup memory/CPU quota, `/dev/shm`, disk, and live E5/GPU processes using the [manual HPC guide](../novelty_integrated/e6/README.md). Do not run E6 GPU workloads while they would disturb E5.

In [ ]:
import subprocess, sys
MODEL_DIR = os.environ.get("DR_CONVNEXT_SNAPSHOT")
SOURCE_ROOT = os.environ.get("DR_MANIFEST_SOURCE_ROOT")
CACHE_ROOT = (OUTPUT / "cache" / DATASET / "decoded_rgb_train_v1") if OUTPUT else None
VAL_CACHE = os.environ.get("DR_E5_VAL_CACHE")
RUN_BASE = (OUTPUT / DATASET / "ldam_ordinal_adjacent_seed42_b32_20") if OUTPUT else None
READY_PATHS = bool(ARTIFACT and OUTPUT and MODEL_DIR and SOURCE_ROOT and DATASET in ("APTOS2019", "DDR"))
BASE_ARGS = (["--dataset", DATASET, "--artifact-root", str(ARTIFACT),
              "--manifest-source-root", SOURCE_ROOT, "--model-dir", MODEL_DIR,
              "--train-cache-root", str(CACHE_ROOT), "--trust-legacy-bin",
              "--micro-batch", "32", "--accumulation", "1", "--seed", "42",
              "--num-workers", "2", "--persistent-workers", "--prefetch-factor", "2", "--pin-memory"]
             + (["--validation-cache-root", VAL_CACHE] if VAL_CACHE else [])) if READY_PATHS else []
def e6_command(action, extra):
    assert READY_PATHS, "Set DR_ARTIFACT_ROOT, DR_E6_OUTPUT_ROOT, DR_CONVNEXT_SNAPSHOT and DR_MANIFEST_SOURCE_ROOT"
    return [sys.executable, "-m", "novelty_integrated.e6.runner", action, *BASE_ARGS, *extra]
print("READY_PATHS", READY_PATHS)
if READY_PATHS:
    print("Estimate:", e6_command("estimate-cache", ["--device", "cpu", "--precision", "fp32"]))

### Cache estimate and build

Inspect the estimate and choose `--cache-max-gib` from actual decoded bytes and a free-space reserve. An absent/incompatible E5 validation cache should be omitted consistently from `BASE_ARGS`; the E6 cache builder never writes E5. `RUN_CACHE_BUILD=False` prevents accidental disk allocation.

In [ ]:
if not READY_PATHS:
    print("SKIPPED_MISSING_DATA: external HPC inputs unavailable")
elif RUN_PREFLIGHT:
    subprocess.run(e6_command("estimate-cache", ["--device","cpu","--precision","fp32"]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: read-only cache estimate")
if READY_PATHS and RUN_CACHE_BUILD:
    subprocess.run(e6_command("prepare-cache", ["--device","cpu","--precision","fp32",
        "--cache-max-gib","100","--cache-reserve-gib","100"]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: train RGB cache population")

### Actual-image input equivalence

The runner compares cached versus direct B1 processing with `torch.equal` for train samples at two epochs, and cached E5 validation tensors versus direct official processing when requested. The check verifies all train/validation source hashes first and writes only a small E6 preflight report outside Git. A smoke, benchmark or train command rejects a missing/mismatched report.

In [ ]:
PREFLIGHT_REPORT = RUN_BASE / "input_preflight" / "input_equivalence.json" if RUN_BASE else None
if READY_PATHS and RUN_PREFLIGHT and CACHE_ROOT and CACHE_ROOT.is_dir():
    subprocess.run(e6_command("verify-inputs", ["--device","cpu","--precision","fp32",
        "--sample-per-grade","2","--output-dir",str(PREFLIGHT_REPORT.parent)]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: actual-image equivalence requires a completed E6 cache")

## 7. Model sanity, bounded A100 benchmark and guarded training

GPU work is manual and sequential after E5 isolation checks. The disposable smoke makes two optimizer steps without a checkpoint. The bounded benchmark times actual train/validation batches and estimates 20-epoch time; it is **not** a completed experiment. Training also requires its matching smoke and benchmark reports, >=8 benchmark batches, 1.3× benchmark GPU memory free, a disk reserve and a 50% runtime safety margin. A short subset can still underestimate checkpoint I/O or variable server load.

In [ ]:
GPU_ARGS = ["--device","cuda:0","--precision","bf16",
            "--input-preflight-report",str(PREFLIGHT_REPORT)] if PREFLIGHT_REPORT else []
if READY_PATHS and RUN_SMOKE:
    subprocess.run(e6_command("smoke", [*GPU_ARGS,"--output-dir",str(RUN_BASE / "smoke")]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: GPU smoke")
if READY_PATHS and RUN_BENCHMARK:
    subprocess.run(e6_command("benchmark", [*GPU_ARGS,"--bench-batches","8",
        "--output-dir",str(RUN_BASE / "benchmark")]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: A100 benchmark")
if READY_PATHS and RUN_TRAINING:
    subprocess.run(e6_command("train", [*GPU_ARGS,
        "--smoke-report",str(RUN_BASE / "smoke" / "smoke.json"),
        "--benchmark-report",str(RUN_BASE / "benchmark" / "benchmark.json"),
        "--max-pilot-seconds","21600","--disk-reserve-gib","100",
        "--output-dir",str(RUN_BASE / "run")]), check=True)
else:
    print("SKIPPED_BY_DEFAULT: full E6 pilot")

## 8. Checkpoints and validation-only results

`last.pt` and `best.pt` are kept only under the external E6 run. SHA sidecars, optimizer/RNG, run configuration, source hashes, manifest/processor/cache identities, full metrics and history support guarded resume. Checkpoint selection is validation macro-F1, then lower ordinary categorical CE, then earliest epoch. No final-test path exists. The next cell reads only summaries if available; it does not load a model or perform inference.

In [ ]:
SUMMARY = RUN_BASE / "run" / "summary.json" if RUN_BASE else None
HISTORY = RUN_BASE / "run" / "history.json" if RUN_BASE else None
if SUMMARY and SUMMARY.is_file() and HISTORY.is_file():
    summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
    history = json.loads(HISTORY.read_text(encoding="utf-8"))
    assert summary["status"] == "VALIDATION_COMPLETE"
    assert summary["final_test_status"] == "NOT_ACCESSED"
    assert len(history) == summary["epochs_completed"] == 20
    best = summary["best"]
    print({"best_epoch": best["epoch"], "validation_macro_f1": best["validation"]["macro_f1"],
           "validation_qwk": best["validation"]["quadratic_weighted_kappa"]})
else:
    history = None
    print("NOT RUN: no complete E6 validation summary/history available")

### Curves, confusion matrix and class-wise results

Plots are generated only from real E6 history. `confusion_matrix` has true grades in rows and predicted grades in columns. Grade 1 undergrading is row 1/column 0; overgrading is row 1/columns 2–4. Inspect Grade 3 recall separately. Macro-F1 alone does not establish minority benefit.

In [ ]:
if history is None:
    print("NOT RUN: no E6 curves or confusion matrix")
else:
    import matplotlib.pyplot as plt
    epochs=[row["epoch"] for row in history]
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    ax[0].plot(epochs,[row["train"]["loss"] for row in history],label="train total")
    ax[0].plot(epochs,[row["validation"]["validation_loss"] for row in history],label="validation CE")
    ax[0].set(xlabel="Epoch",ylabel="Loss",title="Training and validation losses"); ax[0].legend()
    ax[1].plot(epochs,[row["train"]["macro_f1"] for row in history],label="train")
    ax[1].plot(epochs,[row["validation"]["macro_f1"] for row in history],label="validation")
    ax[1].set(xlabel="Epoch",ylabel="Macro-F1",title="Macro-F1 trajectory"); ax[1].legend()
    plt.show()
    best=summary["best"]["validation"]
    cm=best["confusion_matrix"]
    fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,cmap="Blues")
    ax.set(xlabel="Predicted grade",ylabel="True grade",title="Best E6 validation checkpoint")
    ax.set_xticks(range(5)); ax.set_yticks(range(5)); fig.colorbar(im,ax=ax); plt.show()
    print("Per-grade metrics:", best["per_grade"])
    print("Grade 1 correct/total:",cm[1][1],"/",sum(cm[1]))
    print("Grade 1 to G0:",cm[1][0],"; to G2–G4:",sum(cm[1][2:]))

## 9. Interpretation, limitations and final status

E6 status remains **NOT RUN** until verified external artifacts are supplied. No F1 improvement, Grade 1 recovery, throughput gain or clinical mechanism can be inferred from implementation tests. The cache may reduce repeat JPEG decoding, but per-entry integrity hashing and shared disk load can offset savings; measure actual end-to-end throughput and E5 process impact. The short benchmark does not prove a six-hour completion time. LDAM, effective-number weighting, ordinal BCE and adjacent softplus may overlap; preplanned component ablations are needed. An E6-B warm-start would be a different, shorter protocol and is not enabled here. The frozen test split remains sealed during development.